In [1]:
# imports
import json
import numpy as np
import wsidicom
import matplotlib.pyplot as plt
import numpy as np
import cv2
from pathlib import Path
import os
import glob
import shutil

In [2]:
# constants 
GRID_SIZE = 640
BASE_PATH = "RawData/" # used when reading dicom- and geojson-files
numberOfDirectories = 0 # iteration variable based on number of dicom-files
loadedGeojson = False
loadedDicom = False
geoJsonFileName = ""
classes = {}
setOfCoordinatePatches = set()
trainingSplit = 0.8 #80%
#testingSplit = 0.9 #20%
validationSplit = 0.2 #20%
totalNumberOfAnnotationsProcessed = 0
totalNumberOfPatchesSaved = 0

In [3]:
### FUNCTION DEFINITIONS

In [4]:
def createFolderStructureForDataset():
    global mainDirectory
    mainDirectory = "Dataset/"
    
    imagesFolder = mainDirectory + "images/"
    labelsFolder = mainDirectory + "labels/"
    
    global trainingFolderImages
    trainingFolderImages = imagesFolder + "train/"
    
    global testingFolderImages
    testingFolderImages = imagesFolder + "test/"
    
    global validationFolderImages
    validationFolderImages = imagesFolder + "val/"

    global trainingFolderLabels
    trainingFolderLabels = labelsFolder + "train/"
    
    global testingFolderLabels
    testingFolderLabels = labelsFolder + "test/"
    
    global validationFolderLabels
    validationFolderLabels = labelsFolder + "val/"
    # Bad practice, as the first one catches an exception the rest will not be executed -> needs fix later
    try:
        os.makedirs(trainingFolderImages)
        os.makedirs(testingFolderImages)
        os.makedirs(validationFolderImages)
        os.makedirs(trainingFolderLabels)
        os.makedirs(testingFolderLabels)
        os.makedirs(validationFolderLabels)
    except FileExistsError as fee:
        print("Folders are already created")
    except PermissionError as pe:
        print(f'Permission denied, error: {pe}')
    except Exception as e:
        print(f'Error occured: {e}')

In [5]:
def countNumberOfSubfolders():
    count = 0
    path = Path(BASE_PATH)
    for entry in path.iterdir():
        if entry.is_dir():
            count += 1
    return count

In [6]:
def readGeoJSONFile(path, dicomHeight):
    global loadedGeojson
    global geoJsonFileName
    geojsonFile = glob.glob(path + "/*.geojson")
    try:
        with open(geojsonFile[0]) as file:
            contentsOfFile = json.load(file)
            geoJsonFileName = fetchGeoJSONName(geojsonFile)
            findAllClassesAndAddThemToDictionary(contentsOfFile)
            convertFromGeojsonToYoloFormat(contentsOfFile, dicomHeight)
            loadedGeojson = True
    except Exception as e:
        loadedGeojson = False
        print(f'Files not loaded: could not read GeoJSON file from path {path}, error: {e}')

In [7]:
def fetchGeoJSONName(listOfFilesWithGeoJson):
    splitFileFromFolder = listOfFilesWithGeoJson[0].split('\\')
    finalFileName = splitFileFromFolder[1].split('.')
    return finalFileName[0]

In [8]:
def findAllClassesAndAddThemToDictionary(contentsOfGeoJsonFile):
    counter = 0
    global classes

    try:
        for i in range(len(contentsOfGeoJsonFile['features'])):
            classificationType = contentsOfGeoJsonFile['features'][i]['properties']['classification']['name']
            if classificationType not in classes:
                classes[classificationType] = counter
                counter += 1
    except KeyError as ke:
        print("No classes found")

In [9]:
def fetchImageHeight(current_path):
    try:
        with wsidicom.WsiDicom.open(current_path) as file:
            return file.levels[0].size.height
    except Exception as e:
        print(f'Files not loaded: could not read DICOM file from path {path}, error: {e}')

In [10]:
def convertFromGeojsonToYoloFormat(geojsonFile, dicomHeight):
    global totalNumberOfAnnotationsProcessed
    global setOfCoordinatePatches
    setOfCoordinatePatches.clear()
    annotationsKept = 0
    
    numberOfAnnotations = len(geojsonFile['features'])
    
    for i in range(numberOfAnnotations):
        # Find y-coordinates for top and bottom corner
        top_left_corner_y = geojsonFile['features'][i]['geometry']['coordinates'][0][0][1]
        bottom_right_corner_y = geojsonFile['features'][i]['geometry']['coordinates'][0][2][1]
    
        # Find x-coordinates for top and right corner
        top_left_corner_x = geojsonFile['features'][i]['geometry']['coordinates'][0][0][0]
        top_right_corner_x = geojsonFile['features'][i]['geometry']['coordinates'][0][2][0] # Changed 3 -> for this geojson to work, if there is mistake later
        
        # Calculate object center
        x_center = (top_left_corner_x + top_right_corner_x)/2
        y_center = (top_left_corner_y + bottom_right_corner_y)/2
    
        # Calculate which grid cell the annotation belongs to
        grid_cell_y = np.floor(y_center/GRID_SIZE)
        grid_cell_x = np.floor(x_center/GRID_SIZE)
        
        # Object center in relation to each patch
        y_center_in_patch = y_center - ((grid_cell_y) * GRID_SIZE)
        x_center_in_patch = x_center - ((grid_cell_x) * GRID_SIZE)
        
        # Calculate height and width of each annotation box
        height_of_bounding_box = bottom_right_corner_y - top_left_corner_y
        width_of_bounding_box = top_right_corner_x - top_left_corner_x
    
        # Find class of object
        if(len(classes) == 0):
            object_class = 0 #default to 0 if no classes are found
        else:
            object_class = classes.get(geojsonFile['features'][i]['properties']['classification']['name'])

        # Don't write annotation if patch exceeds DICOM image size
        if((top_left_corner_y + GRID_SIZE) > dicomHeight):
            continue
            
        # write to file (or append if exists)
        # YOLO format: class x_center y_center width height
        with open(f'Dataset/{geoJsonFileName}[{grid_cell_y},{grid_cell_x}].txt', 'a+') as f:
            f.write(f'{object_class} {x_center_in_patch/GRID_SIZE} {y_center_in_patch/GRID_SIZE} {width_of_bounding_box/GRID_SIZE} {height_of_bounding_box/GRID_SIZE}\n')

        # save grid coordinates to set
        setOfCoordinatePatches.add((grid_cell_x, grid_cell_y))
        annotationsKept += 1
        
        #print(f'Annotation {i + 1}: \nOriginal center (x, y) = ({x_center}, {y_center}), Center in relation to patch (x, y) = ({x_center_in_patch}, {y_center_in_patch}), \nbelongs to grid cell: [{grid_cell_y},{grid_cell_x}], (height, width) = ({height_of_bounding_box}, {width_of_bounding_box}), object class = {object_class}, {classes.get(object_class)}\n')
    #print(f'Number of patches: {len(setOfCoordinatePatches)}')
    print(f'Total annotations kept: {annotationsKept}')
    totalNumberOfAnnotationsProcessed += annotationsKept

In [11]:
def readDicomSeries(path):
    global loadedDicom
    try:
        with wsidicom.WsiDicom.open(path) as file:
            savePatches(file)
            loadedDicom = True
    except Exception as e:
        loadedDicom = False
        print(f'Files not loaded: could not read DICOM file from path {path}, error: {e}')

In [12]:
def savePatches(slide):
    patchesSaved = 0
    global setOfCoordinatePatches
    global totalNumberOfPatchesSaved
    print(f'Size of slide height: {slide.levels[0].size.height}, width: {slide.levels[0].size.width}')
    # locate and save every patch that contains annotations
    for gridCell in setOfCoordinatePatches:
        #print(f'GridCell[1] * 640 = {gridCell[1]*GRID_SIZE}')
        if(((gridCell[1]*GRID_SIZE) + 640) > slide.levels[0].size.height):
            continue;
        region = slide.read_region(
            location=(gridCell[0]*GRID_SIZE, gridCell[1]*GRID_SIZE), 
            level=0,                 
            size=(GRID_SIZE, GRID_SIZE)
        )
        region.save(f"Dataset/{geoJsonFileName}[{gridCell[1]},{gridCell[0]}].jpg")
        patchesSaved += 1
    print(f'Patches saved: {patchesSaved}')
    totalNumberOfPatchesSaved += patchesSaved

In [13]:
def moveFilesToAppropriateFolders():
    global geoJsonFileName
    global setOfCoordinatePatches
    global mainDirectory
    numberOfPatches = len(setOfCoordinatePatches)
    iterationCounter = 1
    for gridCell in setOfCoordinatePatches:
        sourceFile = f'{mainDirectory}{geoJsonFileName}[{gridCell[1]},{gridCell[0]}]'
        if(iterationCounter <= (numberOfPatches * trainingSplit)):
            try:
                shutil.move(sourceFile + ".txt", trainingFolderLabels)
                shutil.move(sourceFile + ".jpg", trainingFolderImages)
            except FileNotFoundError:
                print(f'Could not find file: {sourceFile}')
            except Exception as e:
                print(f"Error: {e}")
        #elif(iterationCounter <= (numberOfPatches * testingSplit)):
        #    try:
        #        shutil.move(sourceFile + ".txt", testingFolderLabels)
        #        shutil.move(sourceFile + ".jpg", testingFolderImages)
        #    except FileNotFoundError:
        #        print(f'Could not find file: {sourceFile}')
        #    except Exception as e:
        #        print(f"Error: {e}")
        else:
            try:
                shutil.move(sourceFile + ".txt", validationFolderLabels)
                shutil.move(sourceFile + ".jpg", validationFolderImages)
            except FileNotFoundError:
                print(f'Could not find file: {sourceFile}')
            except Exception as e:
                print(f"Error: {e}")

        iterationCounter += 1

In [14]:
def createYAMLFile():
    try:
        with open("dataset.yaml", "w") as file:
            file.write("path: Dataset\n")
            file.write("train: images/train\n")
            file.write("val: images/val\n\n")
            #file.write("test: images/test\n\n")
            file.write("nc: 1\n")
            file.write("names: ['plasma cell']")
    except FileExistsError:
        print("File already exists, overwriting content")

In [15]:
createFolderStructureForDataset()
global totalNumberOfAnnotationsProcessed
for i in range(1, countNumberOfSubfolders() + 2):
    print(f'Iteartion {i}:')
    current_path = BASE_PATH + "dicom" + str(i)
    if(not os.path.exists(current_path)):
        print(f'{current_path} not found, skipping iteration...')
        break
    
    dicomHeight = fetchImageHeight(current_path)
    readGeoJSONFile(current_path, dicomHeight)

    if(loadedGeojson):
        readDicomSeries(current_path)
    
    if(loadedGeojson and loadedDicom):
        print("Both files loaded successfully!\n")
        moveFilesToAppropriateFolders()
        print("Moved files with 80/20 split")
        createYAMLFile()
    else:
        print(f'Skipping iteration for path: {current_path}\n')

Iteartion 1:
No classes found
Total annotations kept: 7129
Size of slide height: 87614, width: 163714
Patches saved: 1121
Both files loaded successfully!

Moved files with 80/20 split
Iteartion 2:
No classes found
Total annotations kept: 1240
Size of slide height: 87614, width: 221234
Patches saved: 985
Both files loaded successfully!

Moved files with 80/20 split
Iteartion 3:
No classes found
Total annotations kept: 1546
Size of slide height: 87614, width: 221234
Patches saved: 1383
Both files loaded successfully!

Moved files with 80/20 split
Iteartion 4:
No classes found
Total annotations kept: 4104
Size of slide height: 75161, width: 204432
Patches saved: 1221
Both files loaded successfully!

Moved files with 80/20 split
Iteartion 5:
No classes found
Total annotations kept: 2266
Size of slide height: 87614, width: 221234
Patches saved: 612
Both files loaded successfully!

Moved files with 80/20 split
Iteartion 6:
No classes found
Total annotations kept: 359
Size of slide height: 87

In [16]:
global totalNumberOfAnnotationsProcessed
global totalNumberOfPatchesSaved
print(f'Total number of annotations kept: {totalNumberOfAnnotationsProcessed}')
print(f'Total number of patches kept: {totalNumberOfPatchesSaved}')

Total number of annotations kept: 17816
Total number of patches kept: 6310
